# KAMP Press Equipment Anomaly Detection

이 notebook은 원래 `KMAP.ipynb` 실험을 GitHub 재현용으로 정리한 버전입니다. 재사용 가능한 전처리·feature engineering·평가 로직은 `src/`로 분리했습니다.


In [1]:
from pathlib import Path
import sys

# Jupyter가 notebooks/ 폴더를 작업 디렉터리로 잡아도 src/를 import할 수 있도록 프로젝트 루트를 등록합니다.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import GAP_THRESHOLD_SECONDS, WINDOW_SIZE
from src.data import load_raw_data, create_segments
from src.features import build_feature_dataset
from src.evaluate import evaluate_repeated_group_cv, summarize

print('PROJECT_ROOT:', PROJECT_ROOT)


PROJECT_ROOT: C:\Users\USER\press-equipment-anomaly-detection


## 1. Load raw data


In [2]:
DATA_DIR = PROJECT_ROOT / 'data'
normal, abnormal = load_raw_data(DATA_DIR)
print('normal:', normal.shape)
print('abnormal:', abnormal.shape)


normal: (20000, 6)
abnormal: (600, 6)


## 2. Time-aware segmentation

인접 `TimeStamp` 간격이 0.11초를 초과하면 새로운 segment로 정의합니다.


In [3]:
normal_seg = create_segments(normal, GAP_THRESHOLD_SECONDS)
abnormal_seg = create_segments(abnormal, GAP_THRESHOLD_SECONDS)
print('normal segments:', normal_seg['segment_id'].nunique())
print('abnormal segments:', abnormal_seg['segment_id'].nunique())


normal segments: 599
abnormal segments: 21


## 3. 1-second non-overlapping windows

10 Hz sampling을 기준으로 10 sample ≈ 1초 window를 생성하고, 각 센서별 mean/std/RMS/min/max를 추출합니다.


In [4]:
feature_df = build_feature_dataset(normal_seg, abnormal_seg, WINDOW_SIZE)
feature_df.head()


,dataset,label,segment_id,window_id,window_in_segment,window_start_time,window_end_time,AI0_Vibration_mean,AI0_Vibration_std,AI0_Vibration_rms,...,AI1_Vibration_std,AI1_Vibration_rms,AI1_Vibration_min,AI1_Vibration_max,AI2_Current_mean,AI2_Current_std,AI2_Current_rms,AI2_Current_min,AI2_Current_max,group_id
0,Normal,0,1,1,1,2022-07-12 00:00:00.019,2022-07-12 00:00:00.919,0.003560,0.059079,0.056160,...,0.170443,0.162006,-0.210895,0.258642,32.290700,163.141877,158.102606,-213.795930,217.004330,Normal_segment_1
1,Normal,0,1,2,2,2022-07-12 00:00:01.019,2022-07-12 00:00:01.919,-0.023480,0.072572,0.072742,...,0.161103,0.155345,-0.226471,0.197857,35.887449,162.221934,158.026166,-206.088810,217.325950,Normal_segment_1
2,Normal,0,1,3,3,2022-07-12 00:00:02.019,2022-07-12 00:00:02.919,-0.027718,0.044534,0.050530,...,0.181460,0.172163,-0.266900,0.266851,-87.445997,121.549468,144.719205,-219.317340,138.879790,Normal_segment_1
3,Normal,0,1,4,4,2022-07-12 00:00:03.019,2022-07-12 00:00:03.919,0.014945,0.095042,0.091395,...,0.169144,0.161140,-0.203526,0.275627,107.827402,92.080783,138.772231,-28.200987,219.931510,Normal_segment_1
4,Normal,0,2,5,1,2022-07-12 00:00:07.224,2022-07-12 00:00:08.124,0.015623,0.063459,0.062197,...,0.150112,0.142987,-0.141934,0.244895,-104.313195,88.233090,133.745329,-213.144690,28.972868,Normal_segment_2


## 4. Repeated group-aware validation

`StratifiedGroupKFold`를 사용해 동일 segment의 window가 train/test에 동시에 들어가는 것을 방지합니다.


In [5]:
window_metrics, segment_metrics = evaluate_repeated_group_cv(feature_df)
display(summarize(window_metrics))
display(summarize(segment_metrics))


,mean,std,min,max
split_random_state,220.800000,635.036622,0.000000,2026.000000
abnormal_precision,0.976030,0.012253,0.960784,1.000000
abnormal_recall,0.915094,0.013342,0.886792,0.924528
abnormal_f1,0.944484,0.008166,0.930693,0.960784
balanced_accuracy,0.957217,0.006630,0.943122,0.962264
pr_auc,0.979440,0.003058,0.974886,0.984884


,mean,std,min,max
split_random_state,220.800000,635.036622,0.000000,2026.0
abnormal_precision,1.000000,0.000000,1.000000,1.0
abnormal_recall,0.994118,0.018602,0.941176,1.0
abnormal_f1,0.996970,0.009583,0.969697,1.0
balanced_accuracy,0.997059,0.009301,0.970588,1.0
false_negative_segment_count,0.100000,0.316228,0.000000,1.0
false_positive_segment_count,0.000000,0.000000,0.000000,0.0


## Notes

- 본 저장소의 대표 성능은 개발 단계 repeated Group CV 결과입니다.
- usable abnormal segment 수가 적어 segment-level 점수는 보조적으로 해석합니다.
- 상세 오류분석(Segment 16/20), SHAP, 모델 비교 및 tuning 의사결정은 README와 `docs/`에 정리했습니다.
